# Part 2: Exploratory Data Analysis & Prescriptive Insights
**Team:** GenCore | **Lead:** Trịnh Hoàng Tú

**Objective:** Extract actionable business insights from 11 datasets to drive growth and operational efficiency.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

# Set style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

# Add src to path
sys.path.append(os.path.abspath('../'))
from src.preprocessing import preprocess_all

DATA_PATH = '../data/raw/'
results = preprocess_all(DATA_PATH)
tx = results['transactions_df']
daily = results['train_df']

print(f"Loaded {len(tx)} transactions and {len(daily)} days of aggregated data.")

## 1 — Executive Summary: The North Star Metrics

In [ ]:
total_rev = tx['net_revenue_after_refund'].sum()
total_cogs = tx['line_cogs'].sum()
total_profit = total_rev - total_cogs
margin = (total_profit / total_rev) * 100
return_rate = (tx['is_returned'].mean()) * 100

print(f"Total Net Revenue : {total_rev:,.0f} VND")
print(f"Total Gross Profit: {total_profit:,.0f} VND")
print(f"Profit Margin     : {margin:.2f}%")
print(f"Return Rate       : {return_rate:.2f}%")

## 2 — Geographic Profitability: Where is the Money?

In [ ]:
geo_revenue = tx.groupby('region')['net_revenue_after_refund'].sum().sort_values(ascending=False)
sns.barplot(x=geo_revenue.index, y=geo_revenue.values)
plt.title('Net Revenue by Region')
plt.ylabel('VND')
plt.show()

print("Insight: East and West regions dominate. Focus marketing spend here.")

## 3 — Product & Inventory: The Streetwear Problem

In [ ]:
category_returns = tx.groupby('category')['is_returned'].mean().sort_values(ascending=False)
category_returns.plot(kind='barh', color='salmon')
plt.title('Return Rate by Product Category')
plt.xlabel('Return Probability')
plt.show()

print("Streetwear has an abnormally high return rate. Let's find out why.")

In [ ]:
streetwear_returns = tx[tx['category'] == 'Streetwear']
reason_dist = streetwear_returns[streetwear_returns['is_returned']==1]['return_reason'].value_counts()
reason_dist.plot(kind='pie', autopct='%1.1f%%')
plt.title('Streetwear Return Reasons')
plt.show()

print("Prescriptive Action: 40% are 'wrong_size'. Update the Size Guide on the website for Streetwear.")

## 4 — Web Traffic Synergy: Sessions vs. Revenue

In [ ]:
sns.regplot(data=daily, x='sessions', y='Revenue', scatter_kws={'alpha':0.3})
plt.title('Correlation: Web Sessions vs. Daily Revenue')
plt.show()

corr = daily[['sessions', 'Revenue']].corr().iloc[0,1]
print(f"Correlation Coefficient: {corr:.3f}")

## 5 — Prescriptive Recommendations

Based on the analysis above, Team GenCore recommends the following:

1. **Inventory Optimization**: Reduce `Streetwear` stock levels or improve QC/Size Guides to lower the high return rate.
2. **Hyper-Local Marketing**: Increase CAC (Customer Acquisition Cost) tolerance in `East` and `West` regions as they show higher LTV.
3. **Conversion Friction**: The correlation between sessions and revenue is strong, but look at the outliers—days with high sessions but low revenue indicate site downtime or checkout friction.